In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [8]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [9]:
from sklearn.model_selection import train_test_split
X = data[["text"]]
y = data["label"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Data Preprocessing

In [32]:
import string
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print(string.punctuation)
print(stopwords.words("english")[100:110])

stop_words = set(stopwords.words("english"))
snowball = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()


!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [11]:
import re
import html

def clean_html_text(text):
    if not isinstance(text, str):
        text = str(text)

    # 1. Remove inline JavaScript
    text = re.sub(r'<script.*?</script>', ' ', text, flags=re.DOTALL | re.IGNORECASE)

    # 2. Remove inline CSS
    text = re.sub(r'<style.*?</style>', ' ', text, flags=re.DOTALL | re.IGNORECASE)

    # 3. Remove HTML comments 
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)

    # 4. Remove remaining HTML tags
    text = re.sub(r'<.*?>', ' ', text, flags=re.DOTALL)

    # 5. Decode HTML entities (&amp; &nbsp; &#39; etc.)
    text = html.unescape(text)

  

    return text

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [12]:

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # 3. Remove special characters (punctuation + anything not a letter/space)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # (this covers string.punctuation and more, e.g. emojis, accents)

    # 4. Remove single characters (letter surrounded by spaces)
    text = re.sub(r'\s+[a-z]\s+', ' ', text)

    # 5. Remove single character at the start of the string
    text = re.sub(r'^[a-z]\s+', ' ', text)

    # 6. Remove single character at the end of the string
    text = re.sub(r'\s+[a-z]$', ' ', text)

    # 7. Collapse multiple spaces to one
    text = re.sub(r'\s+', ' ', text).strip()

    # 8. Remove prefixed 'b' (from bytes stringified as b'...')
    text = re.sub(r"^b\s+", "", text)
    text = re.sub(r"^b'", "", text)
    text = re.sub(r"'$", "", text)

    return text

## Now let's work on removing stopwords
Remove the stopwords.

In [22]:

def remove_stopwords(text):
    tokens = text.split()
    tokens = [snowball.stem(w) for w in tokens if w not in stop_words]
    return " ".join(tokens)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [18]:
def lemmatize_text(text):
    # Break sentence into words
    tokens = word_tokenize(text)

    # Reduce each word to its base form
    # pos='v' means treat as verb → "running" → "run"
    # pos='n' means treat as noun → "cars" → "car"
    lemmas = [lemmatizer.lemmatize(w, pos='v') for w in tokens]
    lemmas = [lemmatizer.lemmatize(w, pos='n') for w in lemmas]

    return " ".join(lemmas)

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [33]:
# Your code

def preprocess(text):
    text = clean_html_text(text)
    text = clean_text(text)
    text = remove_stopwords(text)
    text = lemmatize_text(text)
    return text

X_train_clean = X_train.apply(preprocess)
X_test_clean  = X_test.apply(preprocess)

print(X_train_clean.iloc[0][:400])

regard mr nelson smith kind abl reach oscar huma abedin check pat announc monday bank africaag san pedro bp adama ibrahim mean schedul dear friend compliment guess dear presid fdirector name mr let know today tomorrow work name text length dtype str


## Extra features

In [ ]:
data_train = X_train.copy()
data_train["label"] = y_train

data_val = X_test.copy()
data_val["label"] = y_test

data_test = X_test.copy()
data_test["label"] = y_test

data_train["preprocessed_text"] = data_train["text"].apply(preprocess)
data_val["preprocessed_text"] = data_val["text"].apply(preprocess)


# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,regard mr nelson smith kind repli privat email...,0,0,76
535,I have not been able to reach oscar this am. W...,0,abl reach oscar suppos send pdb receiv,0,0,38
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,huma abedin check pat work jack jake rest also...,0,0,73
557,I can have it announced here on Monday - can't...,0,announc monday today,0,0,20
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,bank africaag san pedro bp san pedro cote ivoi...,1,1,931


## How would work the Bag of Words with Count Vectorizer concept?

In [37]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow = CountVectorizer()

X_train_bow = bow.fit_transform(data_train["preprocessed_text"])
X_val_bow = bow.transform(data_val["preprocessed_text"])

model_bow = MultinomialNB()
model_bow.fit(X_train_bow, data_train["label"])

pred_bow = model_bow.predict(X_val_bow)

accuracy_bow = accuracy_score(data_val["label"], pred_bow)

print("BoW accuracy:", accuracy_bow)

BoW accuracy: 0.94


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf.transform(data_val["preprocessed_text"])

print("Training shape:", X_train_tfidf.shape)
print("Validation shape:", X_val_tfidf.shape)

Training shape: (800, 25886)
Validation shape: (200, 25886)


## And the Train a Classifier?

In [41]:

model_tfidf = MultinomialNB()
model_tfidf.fit(X_train_tfidf, data_train["label"])

pred_tfidf = model_tfidf.predict(X_val_tfidf)

accuracy_tfidf = accuracy_score(
    data_val["label"],
    pred_tfidf
)

print("TF-IDF accuracy:", accuracy_tfidf)

TF-IDF accuracy: 0.905


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [42]:
#Experiment 1 — Bag of Words only

from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()
X_train_bow = bow.fit_transform(data_train["preprocessed_text"])
X_val_bow = bow.transform(data_val["preprocessed_text"])
print("Training shape:", X_train_bow.shape)
print("Validation shape:", X_val_bow.shape)
model_bow = MultinomialNB()

model_bow.fit(
    X_train_bow,
    data_train["label"]
)
y_pred_bow = model_bow.predict(X_val_bow)
accuracy_bow = accuracy_score(
    data_val["label"],
    y_pred_bow
)

print("Bag of Words Accuracy:", accuracy_bow)

Training shape: (800, 25886)
Validation shape: (200, 25886)
Bag of Words Accuracy: 0.94


In [43]:
#Experiment 2 — TF-IDF only
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf.transform(data_val["preprocessed_text"])

print("Training shape:", X_train_tfidf.shape)
print("Validation shape:", X_val_tfidf.shape)
model_tfidf = MultinomialNB()
model_tfidf.fit(X_train_tfidf, data_train["label"])

pred_tfidf = model_tfidf.predict(X_val_tfidf)

accuracy_tfidf = accuracy_score(
    data_val["label"],
    pred_tfidf
)

print("TF-IDF accuracy:", accuracy_tfidf)

Training shape: (800, 25886)
Validation shape: (200, 25886)
TF-IDF accuracy: 0.905


In [45]:
#Experiment 3 — Bag of Words + extra flags (money_mark, suspicious_words, text_len)

extra_train = data_train[
    ["money_mark", "suspicious_words", "text_len"]
].values

extra_val = data_val[
    ["money_mark", "suspicious_words", "text_len"]
].values

from scipy.sparse import hstack

X_train_bow_extra = hstack([
    X_train_bow,
    extra_train
])

X_val_bow_extra = hstack([
    X_val_bow,
    extra_val
])

print("Training shape:", X_train_bow_extra.shape)
print("Validation shape:", X_val_bow_extra.shape)

model_bow_extra = MultinomialNB()

model_bow_extra.fit(
    X_train_bow_extra,
    data_train["label"]
)

y_pred_bow_extra = model_bow_extra.predict(X_val_bow_extra)
accuracy_bow_extra = accuracy_score(
    data_val["label"],
    y_pred_bow_extra
)

print("BoW + Extra Features Accuracy:", accuracy_bow_extra)

Training shape: (800, 25889)
Validation shape: (200, 25889)
BoW + Extra Features Accuracy: 0.805


In [46]:
#Experiment 4 — TF-IDF + Extra Features
X_train_tfidf_extra = hstack([
    X_train_tfidf,
    extra_train
])

X_val_tfidf_extra = hstack([
    X_val_tfidf,
    extra_val
])
print("Training shape:", X_train_tfidf_extra.shape)
print("Validation shape:", X_val_tfidf_extra.shape)

model_tfidf_extra = MultinomialNB()

model_tfidf_extra.fit(
    X_train_tfidf_extra,
    data_train["label"]
)

y_pred_tfidf_extra = model_tfidf_extra.predict(
    X_val_tfidf_extra
)

accuracy_tfidf_extra = accuracy_score(
    data_val["label"],
    y_pred_tfidf_extra
)

print(
    "TF-IDF + Extra Features Accuracy:",
    accuracy_tfidf_extra
)

Training shape: (800, 25889)
Validation shape: (200, 25889)
TF-IDF + Extra Features Accuracy: 0.555


In [47]:
results = pd.DataFrame({
    "Method": [
        "Bag of Words",
        "TF-IDF",
        "Bag of Words + Extra Features",
        "TF-IDF + Extra Features"
    ],
    "Accuracy": [
        accuracy_bow,
        accuracy_tfidf,
        accuracy_bow_extra,
        accuracy_tfidf_extra
    ]
})

print(results)

                          Method  Accuracy
0                   Bag of Words     0.940
1                         TF-IDF     0.905
2  Bag of Words + Extra Features     0.805
3        TF-IDF + Extra Features     0.555
